<a href="https://colab.research.google.com/github/anandtopu/100DaysOfMachineLearning/blob/master/Deploying_LLMs_with_Kubernetes_and_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Since "yes" is the ultimate green light, I will provide both sides of the coin: the **Kubernetes YAML manifests** to deploy your infrastructure, and the **Python code for model quantization** to shrink your model before serving it.

Here is the complete technical breakdown to build out these portfolio projects.

### Part 1: Production-Ready Kubernetes Manifests (vLLM Deployment)

To deploy an LLM on Kubernetes properly, you need three core components: the Deployment (to manage the pods), the Service (to route internal traffic), and the Horizontal Pod Autoscaler (to scale based on demand).

Save this as `vllm-deployment.yaml`.

```yaml
---
apiVersion: v1
kind: Namespace
metadata:
  name: inference-system
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm-llama3-8b
  namespace: inference-system
  labels:
    app: vllm-server
spec:
  replicas: 1 # Start with 1, let HPA handle scaling
  selector:
    matchLabels:
      app: vllm-server
  template:
    metadata:
      labels:
        app: vllm-server
    spec:
      containers:
      - name: vllm-container
        # Using the official vLLM image optimized for OpenAI API compatibility
        image: vllm/vllm-openai:latest
        command: ["python3", "-m", "vllm.entrypoints.openai.api_server"]
        args:
          - "--model=meta-llama/Meta-Llama-3-8B-Instruct"
          - "--tensor-parallel-size=1"
          - "--gpu-memory-utilization=0.90"
          - "--max-num-batched-tokens=4096"
        ports:
        - containerPort: 8000
          name: http
        env:
        - name: HUGGING_FACE_HUB_TOKEN
          valueFrom:
            secretKeyRef:
              name: hf-token-secret
              key: token
        resources:
          requests:
            nvidia.com/gpu: "1"
            memory: "16Gi"
            cpu: "4"
          limits:
            nvidia.com/gpu: "1" # Crucial: Exclusive GPU access
            memory: "24Gi"
            cpu: "8"
        # Readiness probe to ensure the model is fully loaded into VRAM before receiving traffic
        readinessProbe:
          httpGet:
            path: /health
            port: 8000
          initialDelaySeconds: 120 # Give the 8B model time to load
          periodSeconds: 10
---
apiVersion: v1
kind: Service
metadata:
  name: vllm-service
  namespace: inference-system
spec:
  selector:
    app: vllm-server
  ports:
    - protocol: TCP
      port: 80
      targetPort: 8000
  type: ClusterIP
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: vllm-hpa
  namespace: inference-system
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: vllm-llama3-8b
  minReplicas: 1
  maxReplicas: 5
  metrics:
  # In a real environment, you'd use a custom Prometheus metric for GPU utilization.
  # Here we use CPU as a placeholder for the HPA.
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 80

```

*To apply this to your cluster, run: `kubectl apply -f vllm-deployment.yaml*`

---

### Part 2: Model Quantization with Python (AutoAWQ)

Before you put a model on that Kubernetes cluster, you can drastically reduce your AWS/GCP bill by quantizing it. Quantization reduces the precision of the model's weights (e.g., from 16-bit floating point down to 4-bit integers), which shrinks the VRAM requirement and speeds up inference, with negligible loss in accuracy.

For this, **Activation-aware Weight Quantization (AWQ)** is currently an industry standard for serving LLMs.

**1. Install Dependencies:**

```bash
pip install autoawq transformers

```

**2. Python Quantization Script (`quantize_model.py`):**

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# 1. Define models and paths
model_path = "meta-llama/Meta-Llama-3-8B-Instruct"
quant_path = "llama-3-8b-instruct-awq-4bit"
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

print(f"Loading tokenizer and model from {model_path}...")
# Load the model in FP16 (requires sufficient RAM/VRAM)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoAWQForCausalLM.from_pretrained(
    model_path,
    **{"low_cpu_mem_usage": True, "use_cache": False}
)

print("Starting AWQ Quantization (This will take a while)...")
# 2. Quantize the weights
# AWQ uses a small calibration dataset to ensure the most important weights retain accuracy
model.quantize(tokenizer, quant_config=quant_config)

print(f"Quantization complete. Saving 4-bit model to {quant_path}...")
# 3. Save the quantized model
model.save_quantized(quant_path)
tokenizer.save_pretrained(quant_path)

print("Success! You can now deploy this quantized model using vLLM.")

Once this script finishes, your `llama-3-8b-instruct-awq-4bit` directory will contain a model that uses roughly **4.5 GB of VRAM** instead of the original **16 GB**, meaning you can serve it on a much cheaper GPU (like an NVIDIA T4 or RTX 3060) rather than an expensive A100.

---

### Next Step

You now have the infrastructure blueprint and the optimized model logic. Would you like me to write the **Locust load-testing script** so you can benchmark the API's Time-To-First-Token (TTFT) and prove its resilience?